In [1]:
import numpy as np
import pandas as pd
from one.api import ONE
from brainbox.io.one import SessionLoader
from pathlib import Path
import os

/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/alf/files.py:10: FutureWarning: `one.alf.files` will be removed in version 3.0. Use `one.alf.path` instead.
  warnings.warn(


In [2]:
# ============================================================================
# Configuration
# ============================================================================
eid = "5c0c560e-9e1f-45e9-b66e-e4ee7855be84"
base_path = "/media/lenny-aharon/T7/ibl-mouse/ibl-mouse_pose/test_200_MVT_dlc_patch_masking/multiview_transformer_200_0/videos_new"

# Initialize ONE
one = ONE(
    base_url='https://openalyx.internationalbrainlab.org', 
    password='international', 
    silent=True
)

In [3]:
# ============================================================================
# Helper function to load lightning pose data
# ============================================================================
def load_lightning_pose_data(eid, camera_view, paw_name, coord, base_path, one):
    """
    Load lightning pose data with correct timestamp mapping.
    
    Parameters:
    -----------
    eid : str
        Experiment ID
    camera_view : str
        "leftCamera" or "rightCamera"
    paw_name : str
        "pawL" or "pawR"
    coord : str
        "x" or "y"
    base_path : str
        Base path to CSV files
    one : ONE object
        ONE API object
        
    Returns:
    --------
    dict with keys: 'times', 'values', 'skip'
    """
    # Load CSV file
    pred_file = os.path.join(base_path, f"_iblrig_{camera_view}.downsampled.{eid}.csv")
    df = pd.read_csv(pred_file, header=[0,1,2], index_col=0)
    
    # Load camera timestamps from IBL
    sess_loader = SessionLoader(one, eid=eid)
    
    # Map camera view to view name for load_motion_energy
    view_map = {
        "leftCamera": "left",
        "rightCamera": "right"
    }
    view_name = view_map[camera_view]
    
    sess_loader.load_motion_energy(views=[view_name])
    camera_times = sess_loader.motion_energy[camera_view]["times"].to_numpy()
    
    # Direct mapping: CSV frame i -> camera_times[i]
    times = camera_times[:len(df)]
    
    # Extract paw data
    idx = pd.IndexSlice
    paw_df = df.loc[:, idx[:, paw_name, coord]]
    paw = paw_df.iloc[:, 0].to_numpy()
    
    # Ensure alignment
    min_len = min(len(times), len(paw))
    times = times[:min_len]
    paw = paw[:min_len]
    
    return {
        "times": times,
        "values": paw,
        "skip": False,
    }



In [4]:
import numpy as np
import pandas as pd
from one.api import ONE
from brainbox.io.one import SessionLoader
from pathlib import Path

In [5]:
# Initialize ONE
one = ONE(
    base_url='https://openalyx.internationalbrainlab.org', 
    password='international', 
    silent=True
)

# Set your EID and paths
eid = "5c0c560e-9e1f-45e9-b66e-e4ee7855be84"  # Your EID from the error
camera_view = "rightCamera"  # or "leftCamera"
pred_file = f"/media/lenny-aharon/T7/ibl-mouse/ibl-mouse_pose/test_200_MVT_dlc_patch_masking/multiview_transformer_200_0/videos_new/_iblrig_{camera_view}.downsampled.{eid}.csv"

print("=" * 80)
print("STEP 1: Load CSV file")
print("=" * 80)
df = pd.read_csv(pred_file, header=[0,1,2], index_col=0)
print(f"CSV shape: {df.shape}")
print(f"CSV index (first 10): {df.index[:10].tolist()}")
print(f"CSV index (last 10): {df.index[-10:].tolist()}")
print(f"Total frames in CSV: {len(df)}")

# Get frame indices
frame_indices = df.index.to_numpy()
print(f"\nFrame indices range: {frame_indices[0]} to {frame_indices[-1]}")
print(f"Frame indices dtype: {frame_indices.dtype}")

print("\n" + "=" * 80)
print("STEP 2: Try to load camera timestamps from IBL")
print("=" * 80)

camera_times = None
method_used = None


STEP 1: Load CSV file
CSV shape: (100000, 9)
CSV index (first 10): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
CSV index (last 10): [99990, 99991, 99992, 99993, 99994, 99995, 99996, 99997, 99998, 99999]
Total frames in CSV: 100000

Frame indices range: 0 to 99999
Frame indices dtype: int64

STEP 2: Try to load camera timestamps from IBL


In [6]:
# ============================================================================
# Load all lightning pose data
# ============================================================================
print("=" * 80)
print("Loading Lightning Pose Data")
print("=" * 80)

# Define all combinations
pose_configs = [
    ("leftCamera", "pawL", "x", "lightning-pose-left-pawL-x"),
    ("leftCamera", "pawL", "y", "lightning-pose-left-pawL-y"),
    ("leftCamera", "pawR", "x", "lightning-pose-left-pawR-x"),
    ("leftCamera", "pawR", "y", "lightning-pose-left-pawR-y"),
    ("rightCamera", "pawL", "x", "lightning-pose-right-pawL-x"),
    ("rightCamera", "pawL", "y", "lightning-pose-right-pawL-y"),
    ("rightCamera", "pawR", "x", "lightning-pose-right-pawR-x"),
    ("rightCamera", "pawR", "y", "lightning-pose-right-pawR-y"),
]

# Store all loaded data
pose_data = {}

for camera_view, paw_name, coord, target_name in pose_configs:
    print(f"\nLoading: {target_name}")
    print(f"  Camera: {camera_view}, Paw: {paw_name}, Coord: {coord}")
    
    try:
        data = load_lightning_pose_data(eid, camera_view, paw_name, coord, base_path, one)
        pose_data[target_name] = data
        
        print(f"  ✓ Success: {len(data['times'])} timestamps, {len(data['values'])} values")
        print(f"    Time range: {data['times'][0]:.3f} to {data['times'][-1]:.3f} seconds")
        print(f"    Value range: {np.nanmin(data['values']):.2f} to {np.nanmax(data['values']):.2f}")
        print(f"    NaN values: {np.sum(np.isnan(data['values']))} ({100*np.sum(np.isnan(data['values']))/len(data['values']):.1f}%)")
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        pose_data[target_name] = {"times": None, "values": None, "skip": True}

# ============================================================================
# Summary
# ============================================================================
print("\n" + "=" * 80)
print("Summary")
print("=" * 80)
print(f"Loaded {len([k for k, v in pose_data.items() if not v['skip']])} / {len(pose_configs)} pose targets")

# Check alignment across all loaded data
if len(pose_data) > 0:
    loaded_data = {k: v for k, v in pose_data.items() if not v['skip']}
    if loaded_data:
        time_lengths = [len(v['times']) for v in loaded_data.values()]
        print(f"\nTimestamp lengths: {set(time_lengths)}")
        if len(set(time_lengths)) == 1:
            print("✓ All timestamps have consistent lengths")
        else:
            print("⚠ Warning: Timestamp lengths differ across targets")
        
        # Show time ranges
        print("\nTime ranges:")
        for target_name, data in loaded_data.items():
            if data['times'] is not None and len(data['times']) > 0:
                print(f"  {target_name}: {data['times'][0]:.3f} to {data['times'][-1]:.3f} seconds")

# ============================================================================
# Example: Access loaded data
# ============================================================================
print("\n" + "=" * 80)
print("Example: Accessing loaded data")
print("=" * 80)
example_target = "lightning-pose-left-pawL-x"
if example_target in pose_data and not pose_data[example_target]['skip']:
    example_data = pose_data[example_target]
    print(f"\nExample: {example_target}")
    print(f"  Times (first 5): {example_data['times'][:5]}")
    print(f"  Values (first 5): {example_data['values'][:5]}")
    print(f"  Shape: times={example_data['times'].shape}, values={example_data['values'].shape}")

print("\n" + "=" * 80)
print("Done!")
print("=" * 80)
print("All data stored in 'pose_data' dictionary")
print("Keys:", list(pose_data.keys()))

Loading Lightning Pose Data

Loading: lightning-pose-left-pawL-x
  Camera: leftCamera, Paw: pawL, Coord: x


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-05-31"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


  ✓ Success: 100000 timestamps, 100000 values
    Time range: 5.625 to 1671.012 seconds
    Value range: 69.89 to 263.30
    NaN values: 0 (0.0%)

Loading: lightning-pose-left-pawL-y
  Camera: leftCamera, Paw: pawL, Coord: y


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-05-31"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


  ✓ Success: 100000 timestamps, 100000 values
    Time range: 5.625 to 1671.012 seconds
    Value range: 60.26 to 189.30
    NaN values: 0 (0.0%)

Loading: lightning-pose-left-pawR-x
  Camera: leftCamera, Paw: pawR, Coord: x


...

  ✓ Success: 100000 timestamps, 100000 values
    Time range: 5.593 to 670.387 seconds
    Value range: 50.98 to 214.67
    NaN values: 0 (0.0%)

Loading: lightning-pose-right-pawR-x
  Camera: rightCamera, Paw: pawR, Coord: x


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-06-01"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


  ✓ Success: 100000 timestamps, 100000 values
    Time range: 5.593 to 670.387 seconds
    Value range: 134.20 to 300.38
    NaN values: 0 (0.0%)

Loading: lightning-pose-right-pawR-y
  Camera: rightCamera, Paw: pawR, Coord: y


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-06-01"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


  ✓ Success: 100000 timestamps, 100000 values
    Time range: 5.593 to 670.387 seconds
    Value range: 64.91 to 198.36
    NaN values: 0 (0.0%)

Summary
Loaded 8 / 8 pose targets

Timestamp lengths: {100000}
✓ All timestamps have consistent lengths

Time ranges:
  lightning-pose-left-pawL-x: 5.625 to 1671.012 seconds
  lightning-pose-left-pawL-y: 5.625 to 1671.012 seconds
  lightning-pose-left-pawR-x: 5.625 to 1671.012 seconds
  lightning-pose-left-pawR-y: 5.625 to 1671.012 seconds
  lightning-pose-right-pawL-x: 5.593 to 670.387 seconds
  lightning-pose-right-pawL-y: 5.593 to 670.387 seconds
  lightning-pose-right-pawR-x: 5.593 to 670.387 seconds
  lightning-pose-right-pawR-y: 5.593 to 670.387 seconds

Example: Accessing loaded data

Example: lightning-pose-left-pawL-x
  Times (first 5): [5.62490412 5.64153579 5.65820079 5.67486579 5.69149746]
  Values (first 5): [238.79656982 238.76470947 238.68415833 238.72581482 238.62303162]
  Shape: times=(100000,), values=(100000,)

Done!
All da

Verification code for order

In [7]:
import numpy as np
import pandas as pd
from one.api import ONE
from brainbox.io.one import SessionLoader

one = ONE(base_url='https://openalyx.internationalbrainlab.org', 
          password='international', silent=True)
eid = "5c0c560e-9e1f-45e9-b66e-e4ee7855be84"

# Load CSV
pred_file = f"/media/lenny-aharon/T7/ibl-mouse/ibl-mouse_pose/test_200_MVT_dlc_patch_masking/multiview_transformer_200_0/videos_new/_iblrig_rightCamera.downsampled.{eid}.csv"
df = pd.read_csv(pred_file, header=[0,1,2], index_col=0)

# Load camera timestamps
sess_loader = SessionLoader(one, eid=eid)
sess_loader.load_motion_energy(views=["right"])
camera_times = sess_loader.motion_energy["rightCamera"]["times"].to_numpy()

print("=" * 80)
print("Verification: Does the mapping make sense?")
print("=" * 80)

# Check 1: Are CSV indices sequential starting from 0?
csv_indices = df.index.to_numpy()
print(f"\n1. CSV frame indices:")
print(f"   First 10: {csv_indices[:10]}")
print(f"   Last 10: {csv_indices[-10:]}")
print(f"   Sequential from 0? {np.array_equal(csv_indices, np.arange(len(df)))}")
print(f"   ✓ Expected: [0, 1, 2, ..., 99999]")

# Check 2: Are camera timestamps in chronological order?
print(f"\n2. Camera timestamps:")
print(f"   First timestamp: {camera_times[0]:.3f} seconds")
print(f"   Second timestamp: {camera_times[1]:.3f} seconds")
print(f"   Last timestamp: {camera_times[-1]:.3f} seconds")
print(f"   Chronologically ordered? {np.all(np.diff(camera_times) > 0)}")
print(f"   ✓ Expected: Increasing timestamps")

# Check 3: Does the mapping make sense?
times = camera_times[:len(df)]
print(f"\n3. Direct mapping (CSV frame i -> camera_times[i]):")
print(f"   CSV frame 0 -> {times[0]:.3f} seconds")
print(f"   CSV frame 1 -> {times[1]:.3f} seconds")
print(f"   CSV frame 100 -> {times[100]:.3f} seconds")
print(f"   CSV frame 99999 -> {times[-1]:.3f} seconds")
print(f"   Duration: {times[-1] - times[0]:.2f} seconds")

# Check 4: Frame rate consistency
frame_diffs = np.diff(times)
avg_frame_interval = np.mean(frame_diffs)
fps = 1.0 / avg_frame_interval
print(f"\n4. Frame rate analysis:")
print(f"   Average frame interval: {avg_frame_interval*1000:.2f} ms")
print(f"   Effective FPS: {fps:.2f}")
print(f"   Frame interval std: {np.std(frame_diffs)*1000:.2f} ms")
print(f"   ✓ Should be relatively constant (camera sampling rate)")

# Check 5: Does this cover the beginning of the session?
print(f"\n5. Session coverage:")
print(f"   Full session: {camera_times[0]:.3f} to {camera_times[-1]:.3f} seconds ({camera_times[-1] - camera_times[0]:.2f} seconds)")
print(f"   Pose data: {times[0]:.3f} to {times[-1]:.3f} seconds ({times[-1] - times[0]:.2f} seconds)")
print(f"   Coverage: {100 * (times[-1] - times[0]) / (camera_times[-1] - camera_times[0]):.1f}% of session")
print(f"   ✓ Expected: Covers first ~10% of session (first 100k frames)")

# Check 6: Verify with trial times
sess_loader.load_trials()
trials_df = sess_loader.trials
if len(trials_df) > 0:
    first_trial_start = trials_df["stimOn_times"].iloc[0]
    last_trial_end = trials_df["feedback_times"].iloc[-1]
    print(f"\n6. Trial alignment:")
    print(f"   First trial starts: {first_trial_start:.3f} seconds")
    print(f"   Last trial ends: {last_trial_end:.3f} seconds")
    print(f"   Pose data covers first trial? {times[0] <= first_trial_start <= times[-1]}")
    print(f"   Pose data covers last trial? {times[0] <= last_trial_end <= times[-1]}")
    print(f"   ✓ Expected: Covers early trials, not later ones")

print("\n" + "=" * 80)
print("Conclusion:")
print("=" * 80)
if (np.array_equal(csv_indices, np.arange(len(df))) and 
    np.all(np.diff(camera_times) > 0) and
    times[0] == camera_times[0]):
    print("✓ Mapping is CORRECT!")
    print("  CSV frame indices are sequential (0, 1, 2, ...)")
    print("  Camera timestamps are in chronological order")
    print("  Direct mapping (frame i -> camera_times[i]) is valid")
else:
    print("⚠ Check the assumptions above - mapping may need adjustment")

/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-06-01"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


Verification: Does the mapping make sense?

1. CSV frame indices:
   First 10: [0 1 2 3 4 5 6 7 8 9]
   Last 10: [99990 99991 99992 99993 99994 99995 99996 99997 99998 99999]
   Sequential from 0? True
   ✓ Expected: [0, 1, 2, ..., 99999]

2. Camera timestamps:
   First timestamp: 5.593 seconds
   Second timestamp: 5.613 seconds
   Last timestamp: 6555.852 seconds
   Chronologically ordered? True
   ✓ Expected: Increasing timestamps

3. Direct mapping (CSV frame i -> camera_times[i]):
   CSV frame 0 -> 5.593 seconds
   CSV frame 1 -> 5.613 seconds
   CSV frame 100 -> 6.271 seconds
   CSV frame 99999 -> 670.387 seconds
   Duration: 664.79 seconds

4. Frame rate analysis:
   Average frame interval: 6.65 ms
   Effective FPS: 150.42
   Frame interval std: 0.05 ms
   ✓ Should be relatively constant (camera sampling rate)

5. Session coverage:
   Full session: 5.593 to 6555.852 seconds (6550.26 seconds)
   Pose data: 5.593 to 670.387 seconds (664.79 seconds)
   Coverage: 10.1% of session
   

In [10]:
import numpy as np
from one.api import ONE
from brainbox.io.one import SessionLoader

# Initialize ONE
one = ONE(
    base_url='https://openalyx.internationalbrainlab.org', 
    password='international', 
    silent=True
)

eid = "9b528ad0-4599-4a55-9148-96cc1d93fb24"

sess_loader = SessionLoader(one, eid=eid)
sess_loader.load_motion_energy(views=["left", "right"])
left_camera_times = sess_loader.motion_energy["leftCamera"]["times"].to_numpy()
right_camera_times = sess_loader.motion_energy["rightCamera"]["times"].to_numpy()

print("=" * 80)
print("Investigating camera FPS")
print("=" * 80)

for camera_name, camera_times in [("Left", left_camera_times), ("Right", right_camera_times)]:
    print(f"\n{camera_name} Camera:")
    print(f"  Total timestamps: {len(camera_times)}")
    print(f"  Duration: {camera_times[-1] - camera_times[0]:.3f} seconds")
    
    # Calculate FPS from total
    fps_total = (len(camera_times) - 1) / (camera_times[-1] - camera_times[0])
    print(f"  FPS (from total): {fps_total:.2f} Hz")
    
    # Check for duplicates
    unique_times = np.unique(camera_times)
    n_duplicates = len(camera_times) - len(unique_times)
    print(f"  Duplicate timestamps: {n_duplicates}")
    
    # Calculate frame intervals (time differences)
    intervals = np.diff(camera_times)
    print(f"  Frame intervals:")
    print(f"    Min: {intervals.min():.6f} seconds")
    print(f"    Max: {intervals.max():.6f} seconds")
    print(f"    Mean: {intervals.mean():.6f} seconds")
    print(f"    Median: {np.median(intervals):.6f} seconds")
    print(f"    Std: {intervals.std():.6f} seconds")
    
    # Expected interval for 60 FPS
    expected_interval_60fps = 1.0 / 60.0
    print(f"  Expected interval at 60 FPS: {expected_interval_60fps:.6f} seconds")
    
    # Check how many intervals are close to 60 FPS
    close_to_60fps = np.sum(np.abs(intervals - expected_interval_60fps) < 0.001)
    print(f"  Intervals close to 60 FPS (±1ms): {close_to_60fps} / {len(intervals)} ({100*close_to_60fps/len(intervals):.1f}%)")
    
    # Check for very small intervals (might indicate oversampling)
    very_small = np.sum(intervals < 0.01)  # Less than 10ms
    print(f"  Very small intervals (<10ms): {very_small}")
    
    # Show first 20 intervals
    print(f"  First 20 intervals:")
    for i in range(min(20, len(intervals))):
        print(f"    Frame {i}->{i+1}: {intervals[i]:.6f} seconds (FPS: {1.0/intervals[i]:.2f})")
    
    # Check if timestamps are evenly spaced
    if len(unique_times) > 1:
        unique_intervals = np.diff(np.sort(unique_times))
        print(f"  Unique timestamps: {len(unique_times)}")
        print(f"  Unique intervals - Mean: {unique_intervals.mean():.6f}, Std: {unique_intervals.std():.6f}")

print("\n" + "=" * 80)
print("Conclusion")
print("=" * 80)
print("If both cameras should be 60 FPS, check:")
print("1. Are there duplicate timestamps?")
print("2. Are the intervals consistent?")
print("3. Is the calculation method correct?")

/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-05-30"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)
/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-06-01"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


Investigating camera FPS

Left Camera:
  Total timestamps: 245371
  Duration: 4086.432 seconds
  FPS (from total): 60.05 Hz
  Duplicate timestamps: 0
  Frame intervals:
    Min: 0.016632 seconds
    Max: 0.033330 seconds
    Mean: 0.016654 seconds
    Median: 0.016665 seconds
    Std: 0.000037 seconds
  Expected interval at 60 FPS: 0.016667 seconds
  Intervals close to 60 FPS (±1ms): 245369 / 245370 (100.0%)
  Very small intervals (<10ms): 0
  First 20 intervals:
    Frame 0->1: 0.033330 seconds (FPS: 30.00)
    Frame 1->2: 0.016632 seconds (FPS: 60.13)
    Frame 2->3: 0.016665 seconds (FPS: 60.01)
    Frame 3->4: 0.016665 seconds (FPS: 60.01)
    Frame 4->5: 0.016665 seconds (FPS: 60.01)
    Frame 5->6: 0.016632 seconds (FPS: 60.13)
    Frame 6->7: 0.016665 seconds (FPS: 60.01)
    Frame 7->8: 0.016665 seconds (FPS: 60.01)
    Frame 8->9: 0.016632 seconds (FPS: 60.13)
    Frame 9->10: 0.016665 seconds (FPS: 60.01)
    Frame 10->11: 0.016665 seconds (FPS: 60.01)
    Frame 11->12: 0.016

In [11]:
import numpy as np
import pandas as pd
from one.api import ONE
from brainbox.io.one import SessionLoader
import os
import sys

# Add the src directory to path to import the function
sys.path.insert(0, '/home/lenny-aharon/neural_decoding/src')
from utils.ibl_data_utils import _load_lightning_pose_from_csv

# Initialize ONE
one = ONE(
    base_url='https://openalyx.internationalbrainlab.org', 
    password='international', 
    silent=True
)

eid = "9b528ad0-4599-4a55-9148-96cc1d93fb24"
base_path = "/media/lenny-aharon/T7/ibl-mouse/ibl-mouse_pose/test_200_MVT_dlc_patch_masking/multiview_transformer_200_0/videos_new"

print("=" * 80)
print("Testing _load_lightning_pose_from_csv function")
print("=" * 80)

# Test configuration
test_configs = [
    ("leftCamera", "pawL", "x"),
    ("leftCamera", "pawL", "y"),
    ("leftCamera", "pawR", "x"),
    ("leftCamera", "pawR", "y"),
    ("rightCamera", "pawL", "x"),
    ("rightCamera", "pawL", "y"),
    ("rightCamera", "pawR", "x"),
    ("rightCamera", "pawR", "y"),
]

results = {}

for camera_view, paw_name, coord in test_configs:
    print(f"\n{'='*80}")
    print(f"Testing: {camera_view} - {paw_name}-{coord}")
    print(f"{'='*80}")
    
    try:
        # Load the data
        result = _load_lightning_pose_from_csv(one, eid, camera_view, paw_name, coord, base_path)
        
        if result["skip"]:
            print(f"  ✗ Skipped: {result}")
            results[(camera_view, paw_name, coord)] = None
            continue
        
        times = result["times"]
        values = result["values"]
        
        print(f"  ✓ Loaded successfully")
        print(f"    Times shape: {times.shape}")
        print(f"    Values shape: {values.shape}")
        print(f"    Time range: {times[0]:.6f} to {times[-1]:.6f} seconds")
        print(f"    Duration: {times[-1] - times[0]:.3f} seconds")
        print(f"    Value range: {np.nanmin(values):.2f} to {np.nanmax(values):.2f}")
        print(f"    NaN values: {np.sum(np.isnan(values))} ({100*np.sum(np.isnan(values))/len(values):.1f}%)")
        
        # Verify timestamp alignment
        # Check if timestamps are in chronological order
        if not np.all(np.diff(times) > 0):
            print(f"    ⚠ Warning: Timestamps are not strictly increasing!")
        else:
            print(f"    ✓ Timestamps are in chronological order")
        
        # Check FPS
        if len(times) > 1:
            intervals = np.diff(times)
            mean_interval = intervals.mean()
            fps = 1.0 / mean_interval
            print(f"    Effective FPS: {fps:.2f} Hz")
            if 59 < fps < 61:
                print(f"    ✓ FPS is correct (~60 Hz)")
            else:
                print(f"    ⚠ Warning: FPS is {fps:.2f} Hz, expected ~60 Hz")
        
        # Verify mapping makes sense
        # Load camera timestamps directly to compare
        sess_loader = SessionLoader(one, eid=eid)
        view_map = {"leftCamera": "left", "rightCamera": "right"}
        view_name = view_map[camera_view]
        sess_loader.load_motion_energy(views=[view_name])
        raw_camera_times = sess_loader.motion_energy[camera_view]["times"].to_numpy()
        
        print(f"\n    Camera timestamp comparison:")
        print(f"      Raw camera first timestamp: {raw_camera_times[0]:.6f} seconds")
        print(f"      Mapped first timestamp: {times[0]:.6f} seconds")
        print(f"      Difference: {abs(times[0] - raw_camera_times[0]):.6f} seconds")
        
        if abs(times[0] - raw_camera_times[0]) < 0.1:
            print(f"      ✓ First timestamps match (within 100ms)")
        else:
            print(f"      ⚠ Warning: First timestamps differ significantly")
        
        # Check if downsampling happened
        if len(times) < len(raw_camera_times):
            print(f"      ✓ Downsampling applied: {len(raw_camera_times)} -> {len(times)} timestamps")
        else:
            print(f"      No downsampling needed: {len(times)} timestamps")
        
        results[(camera_view, paw_name, coord)] = {
            "times": times,
            "values": values,
            "success": True
        }
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        import traceback
        traceback.print_exc()
        results[(camera_view, paw_name, coord)] = None

print(f"\n{'='*80}")
print("Summary")
print(f"{'='*80}")

successful = sum(1 for r in results.values() if r is not None and r.get("success", False))
print(f"Successfully loaded: {successful} / {len(test_configs)}")

# Check consistency across cameras
if successful == len(test_configs):
    print(f"\n✓ All pose data loaded successfully!")
    
    # Check if left and right cameras have similar time ranges
    left_times = [r["times"] for (cv, _, _), r in results.items() if r and cv == "leftCamera" and r.get("success")]
    right_times = [r["times"] for (cv, _, _), r in results.items() if r and cv == "rightCamera" and r.get("success")]
    
    if left_times and right_times:
        left_start = min(t[0] for t in left_times)
        right_start = min(t[0] for t in right_times)
        left_end = max(t[-1] for t in left_times)
        right_end = max(t[-1] for t in right_times)
        
        print(f"\nTime range comparison:")
        print(f"  Left camera: {left_start:.6f} to {left_end:.6f} seconds")
        print(f"  Right camera: {right_start:.6f} to {right_end:.6f} seconds")
        print(f"  Start difference: {abs(left_start - right_start):.6f} seconds")
        
        if abs(left_start - right_start) < 0.1:
            print(f"  ✓ Both cameras start at similar times")
        else:
            print(f"  ⚠ Warning: Cameras start at different times")

print(f"\n{'='*80}")
print("Verification complete!")
print(f"{'='*80}")

Testing _load_lightning_pose_from_csv function

Testing: leftCamera - pawL-x


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-05-30"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)
/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-06-01"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


Mapped CSV frames 343 to 99999 (session times 5.717s to 1666.650s) to camera timestamps
  ✓ Loaded successfully
    Times shape: (99657,)
    Values shape: (99657,)
    Time range: 5.725727 to 1666.648457 seconds
    Duration: 1660.923 seconds
    Value range: 61.90 to 243.06
    NaN values: 0 (0.0%)
    ⚠ Warning: Timestamps are not strictly increasing!
    Effective FPS: 60.00 Hz
    ✓ FPS is correct (~60 Hz)


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-05-30"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)



    Camera timestamp comparison:
      Raw camera first timestamp: 5.725727 seconds
      Mapped first timestamp: 5.725727 seconds
      Difference: 0.000000 seconds
      ✓ First timestamps match (within 100ms)
      ✓ Downsampling applied: 245371 -> 99657 timestamps

Testing: leftCamera - pawL-y


...


    Camera timestamp comparison:
      Raw camera first timestamp: 5.691464 seconds
      Mapped first timestamp: 5.691464 seconds
      Difference: 0.000000 seconds
      ✓ First timestamps match (within 100ms)
      ✓ Downsampling applied: 614694 -> 99659 timestamps

Testing: rightCamera - pawR-y


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-06-01"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)
/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-05-30"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


Mapped CSV frames 341 to 99999 (session times 5.683s to 1666.650s) to camera timestamps
  ✓ Loaded successfully
    Times shape: (99659,)
    Values shape: (99659,)
    Time range: 5.691464 to 1666.644544 seconds
    Duration: 1660.953 seconds
    Value range: 73.17 to 226.16
    NaN values: 0 (0.0%)
    ⚠ Warning: Timestamps are not strictly increasing!
    Effective FPS: 60.00 Hz
    ✓ FPS is correct (~60 Hz)


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-06-01"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)



    Camera timestamp comparison:
      Raw camera first timestamp: 5.691464 seconds
      Mapped first timestamp: 5.691464 seconds
      Difference: 0.000000 seconds
      ✓ First timestamps match (within 100ms)
      ✓ Downsampling applied: 614694 -> 99659 timestamps

Summary
Successfully loaded: 8 / 8

✓ All pose data loaded successfully!

Time range comparison:
  Left camera: 5.725727 to 1666.648457 seconds
  Right camera: 5.691464 to 1666.644544 seconds
  Start difference: 0.034263 seconds
  ✓ Both cameras start at similar times

Verification complete!


# Below is just code I played with for debugging 

The code below shows the times stamp problem

In [7]:

# Method 1: Motion energy (most reliable)
try:
    print("\nTrying Method 1: Motion Energy...")
    sess_loader = SessionLoader(one, eid=eid)
    sess_loader.load_motion_energy(views=[camera_view.replace("Camera", "").lower()])
    camera_times = sess_loader.motion_energy[camera_view]["times"].to_numpy()
    method_used = "Motion Energy"
    print(f"✓ Success! Loaded {len(camera_times)} timestamps")
    print(f"  Time range: {camera_times[0]:.3f} to {camera_times[-1]:.3f} seconds")
except Exception as e:
    print(f"✗ Failed: {e}")

# Method 2: DLC
if camera_times is None:
    try:
        print("\nTrying Method 2: DLC...")
        dlc_data = one.load_object(eid, camera_view, attribute=["dlc", "times"], collection="alf")
        camera_times = dlc_data.times
        method_used = "DLC"
        print(f"✓ Success! Loaded {len(camera_times)} timestamps")
        print(f"  Time range: {camera_times[0]:.3f} to {camera_times[-1]:.3f} seconds")
    except Exception as e:
        print(f"✗ Failed: {e}")

# Method 3: Lightning Pose
if camera_times is None:
    try:
        print("\nTrying Method 3: Lightning Pose...")
        lp_data = one.load_object(eid, camera_view, attribute=["lightningPose", "times"])
        camera_times = lp_data["times"]
        method_used = "Lightning Pose"
        print(f"✓ Success! Loaded {len(camera_times)} timestamps")
        print(f"  Time range: {camera_times[0]:.3f} to {camera_times[-1]:.3f} seconds")
    except Exception as e:
        print(f"✗ Failed: {e}")

print("\n" + "=" * 80)
print("STEP 3: Map CSV frame indices to camera timestamps")
print("=" * 80)

if camera_times is not None:
    print(f"Method used: {method_used}")
    print(f"Camera timestamps available: {len(camera_times)}")
    print(f"CSV frames: {len(frame_indices)}")
    
    # Check if we have enough timestamps
    n_frames_csv = len(frame_indices)
    max_frames = min(100000, len(camera_times), n_frames_csv)
    
    print(f"\nMapping strategy:")
    print(f"  - CSV has {n_frames_csv} frames")
    print(f"  - Camera has {len(camera_times)} timestamps")
    print(f"  - Will use first {max_frames} frames/timestamps")
    
    # Map frame indices to timestamps
    # Frame 0 -> camera_times[0], Frame 1 -> camera_times[1], etc.
    times = camera_times[:max_frames]
    
    # Ensure matching lengths
    if len(times) > n_frames_csv:
        times = times[:n_frames_csv]
        print(f"  - Truncated timestamps to match CSV length: {len(times)}")
    elif len(times) < n_frames_csv:
        print(f"  ⚠ WARNING: CSV has {n_frames_csv} frames but only {len(times)} timestamps available")
        print(f"  - Will truncate CSV data to {len(times)} frames")
        n_frames_csv = len(times)
        df = df.iloc[:n_frames_csv]
        frame_indices = frame_indices[:n_frames_csv]
    else:
        print(f"  ✓ Perfect match!")
    
    print(f"\nFinal mapping:")
    print(f"  - CSV frame 0 -> timestamp {times[0]:.3f} seconds")
    print(f"  - CSV frame 1 -> timestamp {times[1]:.3f} seconds")
    print(f"  - CSV frame {len(times)-1} -> timestamp {times[-1]:.3f} seconds")
    print(f"  - Total duration: {times[-1] - times[0]:.2f} seconds")
    print(f"  - Effective FPS: {len(times) / (times[-1] - times[0]):.2f}")
    
    # Compare with frame-based calculation
    frame_based_times = frame_indices[:len(times)] / 60.0
    print(f"\nComparison with frame-based calculation (assuming 60 FPS):")
    print(f"  - Frame-based: {frame_based_times[0]:.3f} to {frame_based_times[-1]:.3f} seconds")
    print(f"  - Actual timestamps: {times[0]:.3f} to {times[-1]:.3f} seconds")
    print(f"  - Difference at start: {times[0] - frame_based_times[0]:.3f} seconds")
    print(f"  - Difference at end: {times[-1] - frame_based_times[-1]:.3f} seconds")
    
else:
    print("✗ Could not load camera timestamps from any method!")
    print("  Will need to use frame-based calculation (may cause alignment issues)")
    times = frame_indices / 60.0
    print(f"  Calculated times: {times[0]:.3f} to {times[-1]:.3f} seconds")

print("\n" + "=" * 80)
print("STEP 4: Extract paw data and verify alignment")
print("=" * 80)

# Extract paw data (example for pawR-x)
idx = pd.IndexSlice
paw_df = df.loc[:, idx[:, 'pawR', 'x']]
paw = paw_df.iloc[:, 0].to_numpy()

print(f"Paw data shape: {paw.shape}")
print(f"Times shape: {times.shape}")

# Final length check
min_len = min(len(times), len(paw))
times_final = times[:min_len]
paw_final = paw[:min_len]

print(f"\nFinal aligned data:")
print(f"  - Timestamps: {len(times_final)}")
print(f"  - Paw values: {len(paw_final)}")
print(f"  - Time range: {times_final[0]:.3f} to {times_final[-1]:.3f} seconds")
print(f"  - Paw value range: {np.nanmin(paw_final):.2f} to {np.nanmax(paw_final):.2f}")
print(f"  - NaN values in paw: {np.sum(np.isnan(paw_final))} ({100*np.sum(np.isnan(paw_final))/len(paw_final):.1f}%)")

print("\n" + "=" * 80)
print("STEP 5: Check trial alignment (optional)")
print("=" * 80)

# Load trials to see if timestamps align
try:
    sess_loader = SessionLoader(one, eid=eid)
    sess_loader.load_trials()
    trials_df = sess_loader.trials
    
    print(f"Number of trials: {len(trials_df)}")
    if len(trials_df) > 0:
        first_stim_time = trials_df["stimOn_times"].iloc[0]
        last_feedback_time = trials_df["feedback_times"].iloc[-1]
        
        print(f"First stimOn time: {first_stim_time:.3f} seconds")
        print(f"Last feedback time: {last_feedback_time:.3f} seconds")
        print(f"Pose data time range: {times_final[0]:.3f} to {times_final[-1]:.3f} seconds")
        
        # Check if pose data covers trial times
        if times_final[0] <= first_stim_time <= times_final[-1]:
            print(f"✓ Pose data covers first trial")
        else:
            print(f"⚠ Pose data may not cover first trial")
            
        if times_final[0] <= last_feedback_time <= times_final[-1]:
            print(f"✓ Pose data covers last trial")
        else:
            print(f"⚠ Pose data may not cover last trial")
            
except Exception as e:
    print(f"Could not load trials: {e}")

print("\n" + "=" * 80)
print("Summary")
print("=" * 80)
print(f"✓ CSV loaded: {len(df)} frames")
if camera_times is not None:
    print(f"✓ Camera timestamps loaded: {len(camera_times)} timestamps (method: {method_used})")
    print(f"✓ Mapped to: {len(times_final)} aligned timestamps")
    print(f"✓ Paw data extracted: {len(paw_final)} values")
    print(f"\nReady to use in load_target_behavior function!")
else:
    print(f"✗ Could not load camera timestamps - will need fallback method")


Trying Method 1: Motion Energy...


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-06-01"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


✓ Success! Loaded 985302 timestamps
  Time range: 5.593 to 6555.852 seconds

STEP 3: Map CSV frame indices to camera timestamps
Method used: Motion Energy
Camera timestamps available: 985302
CSV frames: 100000

Mapping strategy:
  - CSV has 100000 frames
  - Camera has 985302 timestamps
  - Will use first 100000 frames/timestamps
  ✓ Perfect match!

Final mapping:
  - CSV frame 0 -> timestamp 5.593 seconds
  - CSV frame 1 -> timestamp 5.613 seconds
  - CSV frame 99999 -> timestamp 670.387 seconds
  - Total duration: 664.79 seconds
  - Effective FPS: 150.42

Comparison with frame-based calculation (assuming 60 FPS):
  - Frame-based: 0.000 to 1666.650 seconds
  - Actual timestamps: 5.593 to 670.387 seconds
  - Difference at start: 5.593 seconds
  - Difference at end: -996.263 seconds

STEP 4: Extract paw data and verify alignment
Paw data shape: (100000,)
Times shape: (100000,)

Final aligned data:
  - Timestamps: 100000
  - Paw values: 100000
  - Time range: 5.593 to 670.387 seconds
  -

Load CSV file
CSV shape: (100000, 9)
CSV frame indices: 0 to 99999

Load camera timestamps from IBL


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-06-01"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


Camera timestamps: 985302 total
Time range: 5.593 to 6555.852 seconds

Direct mapping: CSV frame i -> camera_times[i]
Mapped 100000 frames to timestamps
Time range: 5.593 to 670.387 seconds
Duration: 664.79 seconds
Effective FPS: 150.42

Extract paw data
Times shape: (100000,)
Paw shape: (100000,)
Aligned: True

Comparison: WRONG method vs CORRECT method
WRONG (frame/60): 0.000 to 1666.650 seconds
CORRECT (camera_times): 5.593 to 670.387 seconds
Difference at start: 5.593 seconds
Difference at end: -996.263 seconds

Final result
✓ Use these timestamps: times = camera_times[:len(df)]
✓ No FPS calculation needed - direct mapping!
✓ Ready to use in load_target_behavior function


In [ ]:
import numpy as np
import pandas as pd
from one.api import ONE
from brainbox.io.one import SessionLoader

one = ONE(base_url='https://openalyx.internationalbrainlab.org', 
          password='international', silent=True)
eid = "5c0c560e-9e1f-45e9-b66e-e4ee7855be84"

# Load CSV
pred_file = f"/media/lenny-aharon/T7/ibl-mouse/ibl-mouse_pose/test_200_MVT_dlc_patch_masking/multiview_transformer_200_0/videos_new/_iblrig_rightCamera.downsampled.{eid}.csv"
df = pd.read_csv(pred_file, header=[0,1,2], index_col=0)

# Load camera timestamps
sess_loader = SessionLoader(one, eid=eid)
sess_loader.load_motion_energy(views=["right"])
camera_times = sess_loader.motion_energy["rightCamera"]["times"].to_numpy()

print("=" * 80)
print("Verification: Does the mapping make sense?")
print("=" * 80)

# Check 1: Are CSV indices sequential starting from 0?
csv_indices = df.index.to_numpy()
print(f"\n1. CSV frame indices:")
print(f"   First 10: {csv_indices[:10]}")
print(f"   Last 10: {csv_indices[-10:]}")
print(f"   Sequential from 0? {np.array_equal(csv_indices, np.arange(len(df)))}")
print(f"   ✓ Expected: [0, 1, 2, ..., 99999]")

# Check 2: Are camera timestamps in chronological order?
print(f"\n2. Camera timestamps:")
print(f"   First timestamp: {camera_times[0]:.3f} seconds")
print(f"   Second timestamp: {camera_times[1]:.3f} seconds")
print(f"   Last timestamp: {camera_times[-1]:.3f} seconds")
print(f"   Chronologically ordered? {np.all(np.diff(camera_times) > 0)}")
print(f"   ✓ Expected: Increasing timestamps")

# Check 3: Does the mapping make sense?
times = camera_times[:len(df)]
print(f"\n3. Direct mapping (CSV frame i -> camera_times[i]):")
print(f"   CSV frame 0 -> {times[0]:.3f} seconds")
print(f"   CSV frame 1 -> {times[1]:.3f} seconds")
print(f"   CSV frame 100 -> {times[100]:.3f} seconds")
print(f"   CSV frame 99999 -> {times[-1]:.3f} seconds")
print(f"   Duration: {times[-1] - times[0]:.2f} seconds")

# Check 4: Frame rate consistency
frame_diffs = np.diff(times)
avg_frame_interval = np.mean(frame_diffs)
fps = 1.0 / avg_frame_interval
print(f"\n4. Frame rate analysis:")
print(f"   Average frame interval: {avg_frame_interval*1000:.2f} ms")
print(f"   Effective FPS: {fps:.2f}")
print(f"   Frame interval std: {np.std(frame_diffs)*1000:.2f} ms")
print(f"   ✓ Should be relatively constant (camera sampling rate)")

# Check 5: Does this cover the beginning of the session?
print(f"\n5. Session coverage:")
print(f"   Full session: {camera_times[0]:.3f} to {camera_times[-1]:.3f} seconds ({camera_times[-1] - camera_times[0]:.2f} seconds)")
print(f"   Pose data: {times[0]:.3f} to {times[-1]:.3f} seconds ({times[-1] - times[0]:.2f} seconds)")
print(f"   Coverage: {100 * (times[-1] - times[0]) / (camera_times[-1] - camera_times[0]):.1f}% of session")
print(f"   ✓ Expected: Covers first ~10% of session (first 100k frames)")

# Check 6: Verify with trial times
sess_loader.load_trials()
trials_df = sess_loader.trials
if len(trials_df) > 0:
    first_trial_start = trials_df["stimOn_times"].iloc[0]
    last_trial_end = trials_df["feedback_times"].iloc[-1]
    print(f"\n6. Trial alignment:")
    print(f"   First trial starts: {first_trial_start:.3f} seconds")
    print(f"   Last trial ends: {last_trial_end:.3f} seconds")
    print(f"   Pose data covers first trial? {times[0] <= first_trial_start <= times[-1]}")
    print(f"   Pose data covers last trial? {times[0] <= last_trial_end <= times[-1]}")
    print(f"   ✓ Expected: Covers early trials, not later ones")

print("\n" + "=" * 80)
print("Conclusion:")
print("=" * 80)
if (np.array_equal(csv_indices, np.arange(len(df))) and 
    np.all(np.diff(camera_times) > 0) and
    times[0] == camera_times[0]):
    print("✓ Mapping is CORRECT!")
    print("  CSV frame indices are sequential (0, 1, 2, ...)")
    print("  Camera timestamps are in chronological order")
    print("  Direct mapping (frame i -> camera_times[i]) is valid")
else:
    print("⚠ Check the assumptions above - mapping may need adjustment")

In [7]:
import numpy as np
import pandas as pd
from one.api import ONE
from brainbox.io.one import SessionLoader
import os


# Initialize ONE
one = ONE(
    base_url='https://openalyx.internationalbrainlab.org', 
    password='international', 
    silent=True
)

eid = "5c0c560e-9e1f-45e9-b66e-e4ee7855be84"
base_path = "/media/lenny-aharon/T7/ibl-mouse/ibl-mouse_pose/test_200_MVT_dlc_patch_masking/multiview_transformer_200_0/videos_new"

print("=" * 80)
print("Testing lightning-pose-left-pawL-y implementation")
print("=" * 80)

# Test the exact code pattern from your function
target = "lightning-pose-left-pawL-y"
pred_file = os.path.join(base_path, f"_iblrig_leftCamera.downsampled.{eid}.csv")
print(f"Loading CSV: {pred_file}")

df = pd.read_csv(pred_file, header=[0,1,2], index_col=0)
print(f"✓ CSV loaded: {df.shape}")

# Load camera timestamps
sess_loader = SessionLoader(one, eid=eid)
print(f"\nLoading motion energy for leftCamera...")

# BUG CHECK: You're using "rightCamera" but CSV is "leftCamera"!
# This should be "left" not "rightCamera"
try:
    sess_loader.load_motion_energy(views=["left"])  # Should be "left" not "rightCamera"
    camera_times = sess_loader.motion_energy["leftCamera"]["times"].to_numpy()
    print(f"✓ Camera timestamps loaded: {len(camera_times)} timestamps")
    print(f"  Time range: {camera_times[0]:.3f} to {camera_times[-1]:.3f} seconds")
except Exception as e:
    print(f"✗ Error loading motion energy: {e}")
    print("  Make sure you use 'left' for leftCamera, 'right' for rightCamera")

# Direct mapping
times = camera_times[:len(df)]
print(f"\n✓ Direct mapping: {len(times)} timestamps")
print(f"  Time range: {times[0]:.3f} to {times[-1]:.3f} seconds")

# Extract paw data
idx = pd.IndexSlice
paw_df = df.loc[:, idx[:, 'pawL', 'y']]
paw = paw_df.iloc[:, 0].to_numpy()

print(f"\n✓ Paw data extracted: {len(paw)} values")
print(f"  Paw value range: {np.nanmin(paw):.2f} to {np.nanmax(paw):.2f}")
print(f"  NaN values: {np.sum(np.isnan(paw))} ({100*np.sum(np.isnan(paw))/len(paw):.1f}%)")

# Verify alignment
print(f"\n✓ Alignment check:")
print(f"  Times shape: {times.shape}")
print(f"  Paw shape: {paw.shape}")
print(f"  Aligned: {len(times) == len(paw)}")

# Create behavior dict (like your function does)
beh_dict = {
    "times": times,
    "values": paw,
    "skip": False,
}

print(f"\n✓ Behavior dict created:")
print(f"  times: {len(beh_dict['times'])} elements")
print(f"  values: {len(beh_dict['values'])} elements")
print(f"  skip: {beh_dict['skip']}")

print("\n" + "=" * 80)
print("Testing all camera views and paws")
print("=" * 80)

test_configs = [
    ("leftCamera", "left", "pawL", "x"),
    ("leftCamera", "left", "pawL", "y"),
    ("leftCamera", "left", "pawR", "x"),
    ("leftCamera", "left", "pawR", "y"),
    ("rightCamera", "right", "pawL", "x"),
    ("rightCamera", "right", "pawL", "y"),
    ("rightCamera", "right", "pawR", "x"),
    ("rightCamera", "right", "pawR", "y"),
]

for camera_view, view_name, paw_name, coord in test_configs:
    print(f"\nTesting: {camera_view} - {paw_name}-{coord}")
    try:
        pred_file = os.path.join(base_path, f"_iblrig_{camera_view}.downsampled.{eid}.csv")
        df = pd.read_csv(pred_file, header=[0,1,2], index_col=0)
        
        sess_loader = SessionLoader(one, eid=eid)
        sess_loader.load_motion_energy(views=[view_name])  # Use view_name, not camera_view
        camera_times = sess_loader.motion_energy[f"{camera_view}"]["times"].to_numpy()
        times = camera_times[:len(df)]
        
        idx = pd.IndexSlice
        paw_df = df.loc[:, idx[:, paw_name, coord]]
        paw = paw_df.iloc[:, 0].to_numpy()
        
        # Verify
        assert len(times) == len(paw), f"Length mismatch: {len(times)} vs {len(paw)}"
        print(f"  ✓ Success: {len(times)} aligned timestamps and paw values")
        
    except Exception as e:
        print(f"  ✗ Error: {e}")

print("\n" + "=" * 80)
print("Summary")
print("=" * 80)
print("✓ If all tests pass, your code pattern is correct!")
print("⚠ Remember: use 'left' for leftCamera, 'right' for rightCamera in load_motion_energy")

Testing lightning-pose-left-pawL-y implementation
Loading CSV: /media/lenny-aharon/T7/ibl-mouse/ibl-mouse_pose/test_200_MVT_dlc_patch_masking/multiview_transformer_200_0/videos_new/_iblrig_leftCamera.downsampled.5c0c560e-9e1f-45e9-b66e-e4ee7855be84.csv
✓ CSV loaded: (100000, 9)

Loading motion energy for leftCamera...


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-05-31"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)
(S3) /home/lenny-aharon/Downloads/ONE/openalyx.internationalbrainlab.org/steinmetzlab/Subjects/NR_0028/2023-03-07/001/alf/_ibl_leftCamera.times.npy: 100%|██████████| 3.15M/3.15M [00:00<00:00, 12.7MB/s]
(S3) /home/lenny-aharon/Downloads/ONE/openalyx.internationalbrainlab.org/steinmetzlab/Subjects/NR_0028/2023-03-07/001/alf/#2025-05-31#/leftCamera.ROIMotionEnergy.npy: 100%|██████████| 3.15M/3.15M [00:00<00:00, 20.7MB/s]
/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-05-31"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


✓ Camera timestamps loaded: 393311 timestamps
  Time range: 5.625 to 6555.825 seconds

✓ Direct mapping: 100000 timestamps
  Time range: 5.625 to 1671.012 seconds

✓ Paw data extracted: 100000 values
  Paw value range: 60.26 to 189.30
  NaN values: 0 (0.0%)

✓ Alignment check:
  Times shape: (100000,)
  Paw shape: (100000,)
  Aligned: True

✓ Behavior dict created:
  times: 100000 elements
  values: 100000 elements
  skip: False

Testing all camera views and paws

Testing: leftCamera - pawL-x
  ✓ Success: 100000 aligned timestamps and paw values

Testing: leftCamera - pawL-y


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-05-31"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


  ✓ Success: 100000 aligned timestamps and paw values

Testing: leftCamera - pawR-x


...

  ✓ Success: 100000 aligned timestamps and paw values

Testing: rightCamera - pawR-x


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-06-01"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


  ✓ Success: 100000 aligned timestamps and paw values

Testing: rightCamera - pawR-y


/home/lenny-aharon/anaconda3/envs/decoding/lib/python3.9/site-packages/one/util.py:543: ALFWarning: Multiple revisions: "", "2025-06-01"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


  ✓ Success: 100000 aligned timestamps and paw values

Summary
✓ If all tests pass, your code pattern is correct!
⚠ Remember: use 'left' for leftCamera, 'right' for rightCamera in load_motion_energy
